#- [x] Implement `EdgeLoss` in `loss/edge_loss.py`.
- [x] Update `utils/metrics.py` functions to ensure compatibility.
- [x] Integrate Content, Adversarial, and Edge loss into `model_training_srgan.py`.
- [x] Add SSIM, LPIPS, and GMS metrics calculation to the validation loop.
- [/] Implement the same changes in `notebooks/make_srgan_nb.py` and recreate the notebook.** to dynamically load and process data on the fly.

In [ ]:
import sys
!{sys.executable} -m pip install datasets tqdm opencv-python h5py tensorboard lpips scikit-image pyyaml torchvision

In [ ]:
import os
import time
from pathlib import Path
import cv2
import numpy as np
import torch
import math
from torch import nn, optim
from torch.utils.data import DataLoader, IterableDataset
from torch.utils.tensorboard import SummaryWriter
from torchvision.models import vgg19, VGG19_Weights
from skimage.metrics import peak_signal_noise_ratio as psnr_func
from skimage.metrics import structural_similarity as ssim_func
import torch.nn.functional as F
import lpips
from datasets import load_dataset
from tqdm import tqdm
import logging

logging.basicConfig(level=logging.INFO, format="[%(asctime)s] %(levelname)s: %(message)s")
logger = logging.getLogger("remote_training_srgan")

In [ ]:
from dataclasses import dataclass

@dataclass
class StreamingSRGANTrainingConfig:
    root_dir: Path
    model_path_g: Path
    model_path_d: Path
    hf_dataset_name: str
    hf_dataset_config: str
    patch_size: int
    stride: int
    epochs: int
    batch_size: int
    learning_rate_g: float
    learning_rate_d: float
    normalization: str
    device: str
    log_step: int

training_config = StreamingSRGANTrainingConfig(
    root_dir=Path("artifacts/model_training"),
    model_path_g=Path("artifacts/model_training/srgan_gen.pth"),
    model_path_d=Path("artifacts/model_training/srgan_disc.pth"),
    hf_dataset_name="eugenesiow/Div2k",
    hf_dataset_config="bicubic_x4",
    patch_size=96,
    stride=96,
    epochs=200,
    batch_size=16,
    learning_rate_g=0.0001,
    learning_rate_d=0.0001,
    normalization="minus_one_to_one",
    device="cuda",
    log_step=100
)

os.makedirs(training_config.root_dir, exist_ok=True)
os.makedirs(training_config.root_dir / "logs_srgan", exist_ok=True)

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, 1, 1)
        self.bn1 = nn.BatchNorm2d(channels)
        self.prelu = nn.PReLU()
        self.conv2 = nn.Conv2d(channels, channels, 3, 1, 1)
        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x):
        return x + self.bn2(self.conv2(self.prelu(self.bn1(self.conv1(x)))))

class UpsampleBlock(nn.Module):
    def __init__(self, in_channels, up_scale):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, in_channels * up_scale ** 2, 3, 1, 1)
        self.pixel_shuffle = nn.PixelShuffle(up_scale)
        self.prelu = nn.PReLU()

    def forward(self, x):
        return self.prelu(self.pixel_shuffle(self.conv(x)))

class Generator(nn.Module):
    def __init__(self, scale_factor=4, num_residual_blocks=16):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 64, 9, 1, 4)
        self.prelu = nn.PReLU()
        self.res_blocks = nn.Sequential(*[ResidualBlock(64) for _ in range(num_residual_blocks)])
        self.conv2 = nn.Conv2d(64, 64, 3, 1, 1)
        self.bn2 = nn.BatchNorm2d(64)
        self.upsample_blocks = nn.Sequential(*[UpsampleBlock(64, 2) for _ in range(int(math.log2(scale_factor)))])
        self.conv3 = nn.Conv2d(64, 3, 9, 1, 4)

    def forward(self, x):
        out1 = self.prelu(self.conv1(x))
        out = self.res_blocks(out1)
        out2 = self.bn2(self.conv2(out))
        out = self.upsample_blocks(out1 + out2)
        return torch.tanh(self.conv3(out))

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1), nn.LeakyReLU(0.2),
            nn.Conv2d(64, 64, 3, stride=2, padding=1), nn.BatchNorm2d(64), nn.LeakyReLU(0.2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.LeakyReLU(0.2),
            nn.Conv2d(128, 128, 3, stride=2, padding=1), nn.BatchNorm2d(128), nn.LeakyReLU(0.2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.LeakyReLU(0.2),
            nn.Conv2d(256, 256, 3, stride=2, padding=1), nn.BatchNorm2d(256), nn.LeakyReLU(0.2),
            nn.Conv2d(256, 512, 3, padding=1), nn.BatchNorm2d(512), nn.LeakyReLU(0.2),
            nn.Conv2d(512, 512, 3, stride=2, padding=1), nn.BatchNorm2d(512), nn.LeakyReLU(0.2),
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(512, 1024, 1), nn.LeakyReLU(0.2),
            nn.Conv2d(1024, 1, 1)
        )

    def forward(self, x):
        return self.net(x).view(x.size(0))

class VGGLoss(nn.Module):
    def __init__(self, device):
        super().__init__()
        self.vgg = vgg19(weights=VGG19_Weights.DEFAULT).features[:36].eval().to(device)
        self.loss = nn.MSELoss()
        for param in self.vgg.parameters(): param.requires_grad = False
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(device))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(device))

    def forward(self, input, target):
        input = (input - self.mean) / self.std
        target = (target - self.mean) / self.std
        return self.loss(self.vgg(input), self.vgg(target))

class EdgeLoss(nn.Module):
    def __init__(self, device):
        super(EdgeLoss, self).__init__()
        sobel_x = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]], dtype=torch.float32).view(1, 1, 3, 3)
        sobel_y = torch.tensor([[-1., -2., -1.], [ 0.,  0.,  0.], [ 1.,  2.,  1.]], dtype=torch.float32).view(1, 1, 3, 3)
        self.sobel_x = sobel_x.repeat(3, 1, 1, 1).to(device)
        self.sobel_y = sobel_y.repeat(3, 1, 1, 1).to(device)
        self.l1_loss = nn.L1Loss()

    def forward(self, pred, target):
        pred_x = F.conv2d(pred, self.sobel_x, padding=1, groups=3)
        pred_y = F.conv2d(pred, self.sobel_y, padding=1, groups=3)
        target_x = F.conv2d(target, self.sobel_x, padding=1, groups=3)
        target_y = F.conv2d(target, self.sobel_y, padding=1, groups=3)
        pred_mag = torch.sqrt(pred_x**2 + pred_y**2 + 1e-6)
        target_mag = torch.sqrt(target_x**2 + target_y**2 + 1e-6)
        return self.l1_loss(pred_mag, target_mag)

In [ ]:
class StreamingSRGANDataset(IterableDataset):
    def __init__(self, hf_dataset_stream, patch_size=96, stride=96, normalization='minus_one_to_one', shuffle_buffer=1000):
        super().__init__()
        self.hf_dataset = hf_dataset_stream; self.patch_size = patch_size; self.stride = stride
        self.normalization = normalization; self.shuffle_buffer = shuffle_buffer

    def __iter__(self):
        buffer = []
        for item in self.hf_dataset:
            try:
                hr_img = np.array(item['hr'].convert('RGB'))
                lr_img = np.array(item['lr'].convert('RGB'))
            except:
                continue
                
            if hr_img is None or lr_img is None: continue
            lr_upscaled = cv2.resize(lr_img, (hr_img.shape[1], hr_img.shape[0]), interpolation=cv2.INTER_CUBIC)
            
            for y in range(0, hr_img.shape[0] - self.patch_size + 1, self.stride):
                for x in range(0, hr_img.shape[1] - self.patch_size + 1, self.stride):
                    hr_patch = hr_img[y:y+self.patch_size, x:x+self.patch_size].astype(np.float32)
                    lr_patch = lr_upscaled[y:y+self.patch_size, x:x+self.patch_size].astype(np.float32)
                    
                    if self.normalization == "minus_one_to_one":
                        hr_patch = (hr_patch / 127.5) - 1.0; lr_patch = (lr_patch / 127.5) - 1.0
                    
                    hr_tensor = torch.from_numpy(hr_patch).permute(2, 0, 1)
                    lr_tensor = torch.from_numpy(lr_patch).permute(2, 0, 1)
                    
                    buffer.append((lr_tensor, hr_tensor))
                    if len(buffer) >= self.shuffle_buffer:
                        yield buffer.pop(np.random.randint(0, len(buffer)))
                        
        np.random.shuffle(buffer)
        for p in buffer: yield p

loss_fn_alex = lpips.LPIPS(net='alex')

def calculate_psnr(img1, img2):
    return psnr_func(img1, img2, data_range=255.0)

def calculate_ssim(img1, img2):
    return ssim_func(img1, img2, channel_axis=2, data_range=255.0)

def calculate_lpips(img1_tensor, img2_tensor):
    with torch.no_grad():
        if len(img1_tensor.shape) == 3:
            img1_tensor = img1_tensor.unsqueeze(0); img2_tensor = img2_tensor.unsqueeze(0)
        dist = loss_fn_alex(img1_tensor.cpu(), img2_tensor.cpu())
    return dist.mean().item()

def calculate_edge_fidelity(img1, img2):
    import cv2
    grad1_x = cv2.Sobel(img1, cv2.CV_64F, 1, 0, ksize=3); grad1_y = cv2.Sobel(img1, cv2.CV_64F, 0, 1, ksize=3)
    grad2_x = cv2.Sobel(img2, cv2.CV_64F, 1, 0, ksize=3); grad2_y = cv2.Sobel(img2, cv2.CV_64F, 0, 1, ksize=3)
    mag1 = np.sqrt(grad1_x**2 + grad1_y**2); mag2 = np.sqrt(grad2_x**2 + grad2_y**2)
    return np.mean((mag1 - mag2)**2)

In [ ]:
class SRGANTraining:
    def __init__(self, config: StreamingSRGANTrainingConfig):
        self.config = config
        self.device = torch.device(config.device if torch.cuda.is_available() else "cpu")
        self.writer = SummaryWriter(log_dir=str(self.config.root_dir / "logs_srgan"))
        self.scaler_g = torch.amp.GradScaler('cuda') if torch.cuda.is_available() else None
        self.scaler_d = torch.amp.GradScaler('cuda') if torch.cuda.is_available() else None

    def train(self):
        train_loader = self._get_dataloader('train')
        valid_loader = self._get_dataloader('validation')

        netG = Generator().to(self.device)
        netD = Discriminator().to(self.device)
        
        vgg_loss = VGGLoss(self.device)
        mse_loss = nn.MSELoss()
        bce_loss = nn.BCEWithLogitsLoss()
        edge_loss = EdgeLoss(self.device).to(self.device)
        
        optimizer_g = optim.Adam(netG.parameters(), lr=self.config.learning_rate_g)
        optimizer_d = optim.Adam(netD.parameters(), lr=self.config.learning_rate_d)
        
        best_psnr = 0.0; global_step = 0

        if os.path.exists(self.config.model_path_g) and os.path.exists(self.config.model_path_d):
            logger.info("Loading existing models")
            netG.load_state_dict(torch.load(self.config.model_path_g, map_location=self.device))
            netD.load_state_dict(torch.load(self.config.model_path_d, map_location=self.device))

        for epoch in range(self.config.epochs):
            netG.train(); netD.train()
            epoch_loss_g, epoch_loss_d = 0.0, 0.0
            progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{self.config.epochs}")
            
            for batch_idx, (lr, hr) in enumerate(progress_bar):
                lr, hr = lr.to(self.device), hr.to(self.device)
                
                # Discriminator
                optimizer_d.zero_grad()
                amp_ctx = torch.amp.autocast('cuda') if self.scaler_d else torch.autocast("cpu", enabled=False)
                with amp_ctx:
                    fake_hr = netG(lr)
                    real_pred = netD(hr)
                    fake_pred = netD(fake_hr.detach())
                    d_loss = bce_loss(real_pred, torch.ones_like(real_pred)) + bce_loss(fake_pred, torch.zeros_like(fake_pred))
                
                if self.scaler_d:
                    self.scaler_d.scale(d_loss).backward()
                    self.scaler_d.step(optimizer_d)
                    self.scaler_d.update()
                else: d_loss.backward(); optimizer_d.step()
                
                # Generator
                optimizer_g.zero_grad()
                amp_ctx = torch.amp.autocast('cuda') if self.scaler_g else torch.autocast("cpu", enabled=False)
                with amp_ctx:
                    fake_pred = netD(fake_hr)
                    g_adv_loss = bce_loss(fake_pred, torch.ones_like(fake_pred))
                    g_content_loss = vgg_loss((fake_hr+1)/2, (hr+1)/2)
                    g_edge_loss = edge_loss(fake_hr, hr)
                    g_mse_loss = mse_loss(fake_hr, hr)
                    g_loss = g_mse_loss + 0.006 * g_content_loss + 1e-3 * g_adv_loss + 0.1 * g_edge_loss
                
                if self.scaler_g:
                    self.scaler_g.scale(g_loss).backward()
                    self.scaler_g.step(optimizer_g)
                    self.scaler_g.update()
                else: g_loss.backward(); optimizer_g.step()
                
                epoch_loss_d += d_loss.item(); epoch_loss_g += g_loss.item()
                global_step += 1
                
                if global_step % self.config.log_step == 0:
                    avg_d = epoch_loss_d/(batch_idx+1); avg_g = epoch_loss_g/(batch_idx+1)
                    self.writer.add_scalar("Loss/D", avg_d, global_step)
                    self.writer.add_scalar("Loss/G", avg_g, global_step)
                    progress_bar.set_postfix({"D": f"{avg_d:.4f}", "G": f"{avg_g:.4f}"})
            
            avg_val_loss, avg_psnr, avg_ssim, avg_lpips, avg_gms = self._validate(netG, valid_loader, mse_loss)
            self.writer.add_scalar("Metrics/PSNR_dB", avg_psnr, epoch)
            self.writer.add_scalar("Metrics/SSIM", avg_ssim, epoch)
            self.writer.add_scalar("Metrics/LPIPS", avg_lpips, epoch)
            self.writer.add_scalar("Metrics/GMS", avg_gms, epoch)
            logger.info(f"Epoch {epoch+1}: PSNR={avg_psnr:.2f}dB | SSIM={avg_ssim:.4f} | LPIPS={avg_lpips:.4f} | GMS={avg_gms:.4f}")
            
            if avg_psnr > best_psnr:
                best_psnr = avg_psnr
                torch.save(netG.state_dict(), self.config.model_path_g)
                torch.save(netD.state_dict(), self.config.model_path_d)
        
        self.writer.close()

    def _validate(self, netG, loader, criterion):
        netG.eval(); val_loss = 0.0; psnr_values, ssim_values, lpips_values, gms_values = [], [], [], []
        with torch.no_grad():
            for lr, hr in tqdm(loader, desc="Validation"):
                lr, hr = lr.to(self.device), hr.to(self.device)
                amp_ctx = torch.amp.autocast('cuda') if torch.cuda.is_available() else torch.autocast("cpu", enabled=False)
                with amp_ctx:
                    fake_hr = netG(lr)
                    loss = criterion(fake_hr, hr)
                val_loss += loss.item()
                out_np = np.clip((fake_hr.cpu().numpy() + 1.0) * 127.5, 0, 255).astype(np.uint8)
                hr_np = np.clip((hr.cpu().numpy() + 1.0) * 127.5, 0, 255).astype(np.uint8)
                
                for i in range(out_np.shape[0]):
                    img_out = np.transpose(out_np[i], (1, 2, 0))
                    img_hr = np.transpose(hr_np[i], (1, 2, 0))
                    psnr_values.append(calculate_psnr(img_out, img_hr))
                    ssim_values.append(calculate_ssim(img_out, img_hr))
                    lpips_values.append(calculate_lpips(fake_hr[i:i+1], hr[i:i+1]))
                    
                    import cv2
                    img_out_gray = cv2.cvtColor(img_out, cv2.COLOR_RGB2GRAY)
                    img_hr_gray = cv2.cvtColor(img_hr, cv2.COLOR_RGB2GRAY)
                    gms_values.append(calculate_edge_fidelity(img_out_gray, img_hr_gray))
                    
        return val_loss, np.mean(psnr_values), np.mean(ssim_values), np.mean(lpips_values), np.mean(gms_values)

    def _get_dataloader(self, split):
        logger.info(f"Loading HF dataset split: {split}")
        hf_ds = load_dataset(self.config.hf_dataset_name, name=self.config.hf_dataset_config, split=split, streaming=True)
        dataset = StreamingSRGANDataset(
            hf_ds,
            patch_size=self.config.patch_size,
            stride=self.config.stride,
            normalization=self.config.normalization,
            shuffle_buffer=2000 if split == 'train' else 10
        )
        return DataLoader(dataset, batch_size=self.config.batch_size, pin_memory=True)

In [ ]:
logger.info("--- Starting SRGAN Model Training (Streaming Mode) ---")
mt = SRGANTraining(training_config)
mt.train()